# 21.2 First-order logic

Variables and quantifiers let logic talk about objects, relations, and reusable rules instead of one flat proposition at a time.

First-order logic keeps truth conditions but lets statements mention objects. Pattern matching becomes unification, which powers rules, theorem proving, and relational inference.

Save a copy to Drive to edit.

In [ ]:

import itertools
import random

import matplotlib.pyplot as plt

random.seed(2102)


def is_variable(term):
    return isinstance(term, str) and term.startswith("?")


def resolve(term, env):
    while is_variable(term) and term in env:
        term = env[term]
    return term


def unify_terms(left, right, env):
    left_value = resolve(left, env)
    right_value = resolve(right, env)
    new_env = dict(env)
    if left_value == right_value:
        return new_env
    if is_variable(left_value):
        new_env[left_value] = right_value
        return new_env
    if is_variable(right_value):
        new_env[right_value] = left_value
        return new_env
    return None


def unify_atom(pattern, fact, env):
    if pattern[0] != fact[0]:
        return None
    if len(pattern) != len(fact):
        return None
    new_env = dict(env)
    for left, right in zip(pattern[1:], fact[1:]):
        new_env = unify_terms(left, right, new_env)
        if new_env is None:
            return None
    return new_env


def substitute(atom, env):
    return tuple(resolve(part, env) for part in atom)


def match_body(body, facts):
    envs = [dict()]
    match_count = 0
    for atom in body:
        next_envs = []
        for env in envs:
            for fact in facts:
                match_count += 1
                new_env = unify_atom(atom, fact, env)
                if new_env is not None:
                    next_envs.append(new_env)
        envs = next_envs
    return envs, match_count


def fol_forward_chain(domain, predicates, rules, max_rounds=20):
    facts = set(predicates)
    trace = []
    total_matches = 0
    for _ in range(max_rounds):
        added = []
        for rule in rules:
            envs, match_count = match_body(rule["body"], facts)
            total_matches += match_count
            for env in envs:
                head = substitute(rule["head"], env)
                if head not in facts:
                    facts.add(head)
                    added.append(head)
                    trace.append((rule["name"], dict(env), head))
        if not added:
            break
    return {"facts": facts, "trace": trace, "matches": total_matches, "domain": list(domain)}


def make_fol_ladder():
    d1_domain = ["Tweety", "Opus", "Fido"]
    d1_facts = {("Bird", "Tweety"), ("Bird", "Opus"), ("Wings", "Tweety"), ("Wings", "Opus"), ("Flies", "Tweety")}
    d1_rules = [{"name": "birds_have_wings", "body": [("Bird", "?x")], "head": ("Wings", "?x")}]
    d2_domain = d1_domain + ["Ann", "Bob", "Cal"]
    d2_facts = set(d1_facts) | {("Parent", "Ann", "Bob"), ("Parent", "Bob", "Cal")}
    d2_rules = d1_rules + [{"name": "grandparent", "body": [("Parent", "?x", "?y"), ("Parent", "?y", "?z")], "head": ("Grandparent", "?x", "?z")}]
    d3_domain = d2_domain + ["Ghost"]
    d3_facts = set(d2_facts) | {("Bird", "Ghost")}
    d3_rules = d2_rules + [{"name": "winged_can_be_seen", "body": [("Wings", "?x")], "head": ("Visible", "?x")}]
    d4_domain = [f"User{i}" for i in range(20)]
    d4_facts = {("Employee", name) for name in d4_domain}
    d4_facts |= {("Admin", name) for name in d4_domain[:6]}
    d4_rules = [{"name": "admin_deploy", "body": [("Admin", "?x")], "head": ("CanDeploy", "?x")}, {"name": "employee_badge", "body": [("Employee", "?x")], "head": ("HasBadge", "?x")}]
    d5_domain = [f"Person{i}" for i in range(50)]
    d5_facts = {("Link0", "Person0", "Person1")}
    for i in range(1, 5):
        d5_facts.add((f"Link{i}", f"Person{i}", f"Person{i + 1}"))
    d5_facts |= {("Admin", f"Person{i}") for i in range(12)}
    d5_rules = [
        {"name": "hop01", "body": [("Link0", "?a", "?b"), ("Link1", "?b", "?c")], "head": ("Reach2", "?a", "?c")},
        {"name": "hop23", "body": [("Reach2", "?a", "?c"), ("Link2", "?c", "?d")], "head": ("Reach3", "?a", "?d")},
        {"name": "hop34", "body": [("Reach3", "?a", "?d"), ("Link3", "?d", "?e")], "head": ("Reach4", "?a", "?e")},
        {"name": "hop45", "body": [("Reach4", "?a", "?e"), ("Link4", "?e", "?f")], "head": ("Reach5", "?a", "?f")},
        {"name": "admin_deploy", "body": [("Admin", "?x")], "head": ("CanDeploy", "?x")},
    ]
    return [
        {"name": "D1 birds", "domain": d1_domain, "facts": d1_facts, "rules": d1_rules, "target": ("Wings", "Tweety")},
        {"name": "D2 taxonomy relation", "domain": d2_domain, "facts": d2_facts, "rules": d2_rules, "target": ("Grandparent", "Ann", "Cal")},
        {"name": "D3 drift and constants", "domain": d3_domain, "facts": d3_facts, "rules": d3_rules, "target": ("Visible", "Ghost")},
        {"name": "D4 organization", "domain": d4_domain, "facts": d4_facts, "rules": d4_rules, "target": ("CanDeploy", "User5")},
        {"name": "D5 family security chain", "domain": d5_domain, "facts": d5_facts, "rules": d5_rules, "target": ("Reach5", "Person0", "Person5")},
    ]


## The concept, built once (D1)

A universal rule such as $orall x\, Bird(x)	o Wings(x)$ becomes one reusable rule with a variable. Unification binds $?x$ to each matching object and forward chaining adds the grounded head.

In [ ]:

domain = ["Tweety", "Opus", "Fido"]
facts = {("Bird", "Tweety"), ("Bird", "Opus"), ("Wings", "Tweety"), ("Wings", "Opus"), ("Flies", "Tweety")}
rules = [{"name": "birds_have_wings", "body": [("Bird", "?x")], "head": ("Wings", "?x")}]

d1 = fol_forward_chain(domain, facts, rules)
bird_wing_rows = [(obj, ("Bird", obj) in facts, ("Wings", obj) in d1["facts"]) for obj in domain]
witnesses = [obj for obj in domain if ("Bird", obj) in facts and ("Flies", obj) in facts]

print(bird_wing_rows)
print("witnesses", witnesses)


The lesson's worked structure has $3$ objects. The universal rule is true for all $3$ rows, and $\exists x\,Bird(x)\land Flies(x)$ has exactly $1$ witness: Tweety.

In [ ]:

assert len(domain) == 3
assert all((not is_bird) or has_wings for _, is_bird, has_wings in bird_wing_rows)
assert len(witnesses) == 1
assert witnesses[0] == "Tweety"


## The dataset ladder

The inline D1-D5 ladder grows from three named objects to a 50-object security and family-style chain with a four-hop relational inference.

In [ ]:

fol_ladder = make_fol_ladder()

for rung in fol_ladder:
    print(rung["name"], "domain", len(rung["domain"]), "facts", len(rung["facts"]), "rules", len(rung["rules"]), "sample", list(rung["facts"])[:4])


## Run the same method across D1-D5

Each rung uses the same unification and forward-chaining engine. The metric records whether the target fact is derived and how many ground-rule matches were attempted.

In [ ]:

fol_results = []
for rung in fol_ladder:
    result = fol_forward_chain(rung["domain"], rung["facts"], rung["rules"])
    correct = rung["target"] in result["facts"]
    fol_results.append({
        "name": rung["name"],
        "size": len(rung["domain"]),
        "derived": len(result["facts"] - rung["facts"]),
        "matches": result["matches"],
        "correct": correct,
        "trace": result["trace"],
    })

for row in fol_results:
    print(row["name"], row["size"], row["derived"], row["matches"], row["correct"])


## Results visualization

The panels display inference traces as proof chains. The curve relates domain size to the number of unification attempts.

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
flat_axes = axes.ravel()

for ax, row in zip(flat_axes[:5], fol_results):
    trace_text = "\n".join(f"{name} -> {head}" for name, _, head in row["trace"][:5])
    ax.text(0.03, 0.95, trace_text or "no new facts", va="top", fontsize=8)
    ax.set_title(row["name"])
    ax.axis("off")

ax = flat_axes[5]
ax.plot([row["size"] for row in fol_results], [row["matches"] for row in fol_results], marker="o")
ax.set_xlabel("domain size")
ax.set_ylabel("unification attempts")
ax.set_title("match cost vs domain")
plt.tight_layout()


## Pitfall on the hardest rung

Pitfall: assuming existence from a universal rule or letting the domain drift. On D5, a permission rule says what follows for explicit admins; it does not prove that some new admin exists.

In [ ]:

hard = fol_ladder[-1]
hard_result = fol_forward_chain(hard["domain"], hard["facts"], hard["rules"])
wrong_witness = "Person49" in [fact[1] for fact in hard_result["facts"] if fact[0] == "CanDeploy"]
fixed_witnesses = sorted(fact[1] for fact in hard_result["facts"] if fact[0] == "CanDeploy")

print("wrong assumed witness Person49", wrong_witness)
print("explicit witnesses", fixed_witnesses)


## Evaluate it + Practice

- Metric: target-fact correctness and number of ground-rule matches, compared with a no-skill baseline that guesses or accepts the first candidate.
- Sanity check: rerun D1 and verify the hand-counted lesson numbers before trusting larger rungs.
- Ablation: turn off the key symbolic check and confirm the metric drops on D3-D5.
- Failure signals: unexplained answers, fewer checked cases than the rung size requires, or a contradiction accepted as valid.
- Keep all instances CPU-only, seeded, and small enough to inspect.

Practice: Add one Parent chain to D2 and derive a new Grandparent fact.

Practice: Change Ghost from a constant into ?Ghost and explain the unification bug.

Practice: Remove all Admin facts in D5 and state what the universal rule can and cannot prove.